In [ ]:
%jars pg/pg-api/build/libs/pg-api-1.0.0-SNAPSHOT.jar 
%jars pg/pg-global/build/libs/pg-global-1.0.0-SNAPSHOT.jar
%jars pg/pg-multiverse/build/libs/pg-multiverse-1.0.0-SNAPSHOT.jar
%jars pg/pg-io/build/libs/pg-io-1.0.0-SNAPSHOT.jar

In [ ]:
import java.io.File;
import java.io.IOException;
import java.nio.channels.FileChannel;
import java.nio.file.StandardOpenOption;

import dev.chpg.pg.api.AttributeValue;
import dev.chpg.pg.global.GlobalGraph;
import dev.chpg.pg.io.DirectGraphBufferReader;

In [ ]:
long start = System.currentTimeMillis();
File file = new File(new File("data"), "xinu.dgb");
GlobalGraph targetGraph = new GlobalGraph();
try (FileChannel channel = FileChannel.open(file.toPath(), StandardOpenOption.READ)) {
    DirectGraphBufferReader.read(channel, targetGraph, targetGraph.factory(), targetGraph.factory());
}
long stop = System.currentTimeMillis();
System.out.println("Time: " + (stop-start));

System.out.println("Nodes: " + targetGraph.nodes().size());
System.out.println("Edges: " + targetGraph.edges().size());

System.out.println(
    targetGraph.nodes()
        .withAnyTag("XCSG.Function")
        .withAttribute("XCSG.name", AttributeValue.value("freebuf"))
        .toString()
);

System.out.println(
    targetGraph.nodes()
        .withAnyTag("XCSG.Function")
        .withAttribute("XCSG.name", AttributeValue.value("freebuf"))
        .one().get().tags().toString()
);

System.out.println(
    targetGraph.edges()
        .withAnyTag("XCSG.Call")
        .size()
);

In [ ]:
import dev.chpg.pg.io.DirectGraphBufferReader;
import dev.chpg.pg.multiverse.universe.*;
import dev.chpg.pg.multiverse.ephemeral.*;
import dev.chpg.pg.api.NodeFactory;
import dev.chpg.pg.api.EdgeFactory;
import java.nio.channels.FileChannel;
import java.nio.file.Paths;
import java.nio.file.StandardOpenOption;

// 1. Initialize the empty baseline
Universe universe = new Universe();

// 2. Define factories that strictly generate baseline elements (positive IDs)
NodeFactory universeNodeFactory = () -> {
    int id = universe.idGenerator().createNodeId();
    return new UniverseNode(universe, id);
};

EdgeFactory universeEdgeFactory = (source, target) -> {
    int id = universe.idGenerator().createEdgeId();
    return new UniverseEdge(universe, id, (UniverseNode) source, (UniverseNode) target);
};

// 3. Stream the .dgb directly into the Universe arrays
try (FileChannel channel = FileChannel.open(Paths.get("xinu.dgb"), StandardOpenOption.READ)) {
    DirectGraphBufferReader.read(channel, universe, universeNodeFactory, universeEdgeFactory);
}

// 4. Wrap the populated baseline in the transaction layer for analysis
EphemeralGraph sandbox = new EphemeralGraph(universe);